# Pac-Man DQN 学習 (Google Colab)

1. メニュー「ランタイム → ランタイムのタイプを変更」で GPU を選択
2. 上から順にセルを実行

モデル・ログ・学習曲線はすべて Google Drive の `OUT_DIR` に保存されます。
セッションが切れても、もう一度上から実行すれば `--resume` で続きから再開します。

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== 設定 =====
REPO_URL   = 'https://github.com/HirokiMorita333/MediaDesignTeam1.git'
BRANCH     = 'morita'
CODE_DIR   = '/content/Pacman'                       # コードの置き場所
OUT_DIR    = '/content/drive/MyDrive/Pacman_runs/ghost1'  # 学習結果の保存先 (Drive)
MODEL      = 'pacman_model_1ghost.pt'
EPISODES   = 20000
EXTRA_ARGS = ''   # 例: '--curriculum' / '--ghost-mode bfs'

# GitHub を使わず Drive にコードを置く場合は、そのフォルダを指定 (空文字なら git clone)
DRIVE_CODE_DIR = ''   # 例: '/content/drive/MyDrive/Pacman'

In [ ]:
import os, shutil
if os.path.exists(CODE_DIR):
    shutil.rmtree(CODE_DIR)
if DRIVE_CODE_DIR:
    shutil.copytree(DRIVE_CODE_DIR, CODE_DIR)
else:
    # private リポジトリの場合は URL を https://<TOKEN>@github.com/... にする
    !git clone -b {BRANCH} --depth 1 {REPO_URL} {CODE_DIR}
os.makedirs(OUT_DIR, exist_ok=True)
!ls {CODE_DIR}

In [ ]:
# torch は Colab にプリインストール済みなので残りだけ入れる
!pip install -q gymnasium pygame

In [ ]:
# 動作確認: ゴースト数と観測サイズ
import sys
sys.path.insert(0, CODE_DIR)
from pacman_env import PacManEnv
e = PacManEnv()
obs, _ = e.reset(seed=0)
print('ghosts:', len(e.ghosts), ' obs:', obs.shape)
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
# 学習 (Drive 上の OUT_DIR で実行 → 中間モデル・CSV・学習曲線も Drive に残る)
%cd {OUT_DIR}
resume = '--resume' if os.path.exists(MODEL) else ''
print('resume:', bool(resume))
!python -u {CODE_DIR}/train.py --episodes {EPISODES} --model {MODEL} {resume} {EXTRA_ARGS} 2>&1 | tee -a train_output.txt

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(OUT_DIR, 'learning_curve.png')))
!ls -lh {OUT_DIR}